# Class 16: Compared to What?
### Normal Distributions, the Central Limit Theorem, and Standardization

Work through this notebook with your team. **Record your answers on the paper handout, not here.**

The notebook will tell you when to stop and answer a question.

In [ ]:
import numpy as np
from datascience import *
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')
from scipy.stats import norm

print("Ready!")

---
## Part 1. The Wrong Outlier

39 samples collected along Pennypack Creek on March 22, 2022.

In [ ]:
pp = Table.read_table('data/Pennypack_anions_2022-03-22.csv')
pp.show(5)

In [ ]:
cl = pp.column('Cl (mg/L)')
cl_mean = np.mean(cl)
cl_std  = np.std(cl)

print(f"Mean chloride:  {cl_mean:.2f} mg/L")
print(f"Std deviation:  {cl_std:.2f} mg/L")

Fill in the blank to standardize every chloride measurement.

In [ ]:
z_scores = ...

pp = pp.with_column('Cl z-score', z_scores)
pp.select('SiteCode', 'km downstream', 'Cl (mg/L)', 'Cl z-score').show(5)

In [ ]:
# Which sites does the 2-standard-deviation rule flag?
pp.where('Cl z-score', lambda z: abs(z) > 2).select('SiteCode', 'km downstream', 'Cl (mg/L)', 'Cl z-score')

> **Handout Q1.1**

In [ ]:
# The three highest chloride samples
pp.sort('Cl (mg/L)', descending=True).select('SiteCode', 'km downstream', 'Cl (mg/L)', 'Cl z-score').show(3)

> **Handout Q1.2**, then **Q1.3 — commit to an answer in writing before running the next cell.**

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(pp.column('km downstream'), pp.column('Cl (mg/L)'), color='steelblue', s=40)

for site in ['PP1-1', 'PP1-2', 'PP1-9']:
    row = pp.where('SiteCode', site)
    plt.annotate(site, (row.column('km downstream')[0], row.column('Cl (mg/L)')[0]),
                 textcoords="offset points", xytext=(8, 4))

plt.axhline(cl_mean, color='k', lw=1, label='mean')
plt.axhline(cl_mean + 2*cl_std, color='k', linestyle='--', lw=1, label='mean ± 2 SD')
plt.axhline(cl_mean - 2*cl_std, color='k', linestyle='--', lw=1)
plt.xlabel('km downstream'); plt.ylabel('Cl (mg/L)')
plt.title('Chloride Along Pennypack Creek'); plt.legend(); plt.show()

> **Handout Q1.4 and Q1.5**

### A better comparison group

The creek changes character at the treatment plant near km 8. Standardize each half separately.

In [ ]:
upstream   = pp.where('km downstream', are.below(8))
downstream = pp.where('km downstream', are.above_or_equal_to(8))

def group_z(t, column):
    """Add a z-score column computed within this group only."""
    values = t.column(column)
    z = ...
    return t.with_column('z (within group)', z)

upstream_z   = group_z(upstream, 'Cl (mg/L)')
downstream_z = ...

print(f"Upstream:   n = {upstream.num_rows},  mean = {np.mean(upstream.column('Cl (mg/L)')):.1f} mg/L")
print(f"Downstream: n = {downstream.num_rows},  mean = {np.mean(downstream.column('Cl (mg/L)')):.1f} mg/L")

In [ ]:
print("UPSTREAM of the treatment plant:")
upstream_z.where('z (within group)', lambda z: abs(z) > 2).select('SiteCode', 'Cl (mg/L)', 'z (within group)').show()

print("DOWNSTREAM of the treatment plant:")
downstream_z.where('z (within group)', lambda z: abs(z) > 2).select('SiteCode', 'Cl (mg/L)', 'z (within group)').show()

> **Handout Q1.6, Q1.7, and Q1.8**

---
## Part 2. Many Means

Your 39 samples came from one morning. If the class walked the creek again, how much would the **average** move?

A nitrate logger below the treatment plant recorded every hour for two weeks.

In [ ]:
nitrate_df = pd.read_csv('./data/WWTP_N.csv', usecols=[0, 1], skiprows=1)
nitrate = nitrate_df['Nitrate (mg/l as N)'].values
nitrate = nitrate[~np.isnan(nitrate)]

print(f"Number of hourly readings: {len(nitrate)}")
print(f"Mean:               {np.mean(nitrate):.3f} mg/L")
print(f"Standard deviation: {np.std(nitrate):.3f} mg/L")

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(nitrate, bins=25, density=True, color='seagreen', alpha=0.7, edgecolor='white')

x = np.linspace(np.min(nitrate), np.max(nitrate), 200)
plt.plot(x, norm(loc=np.mean(nitrate), scale=np.std(nitrate)).pdf(x), color='k', lw=2)

plt.xlabel('Nitrate-N (mg/L)'); plt.ylabel('Density')
plt.title('All 330 Hourly Nitrate Readings'); plt.show()

> **Handout Q2.1**

Treat those 330 readings as a **population**. Draw a sample of size $n$, take its mean, and repeat 2,000 times.

In [ ]:
def sample_means(population, n, repetitions=2000):
    means = []
    for i in np.arange(repetitions):
        means.append(np.mean(np.random.choice(population, n)))
    return np.array(means)

np.random.seed(16)   # everyone sees the same result

Fill in three sample sizes: 1, 6, and 24.

In [ ]:
sample_sizes = ...

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, n in zip(axes, sample_sizes):
    ax.hist(sample_means(nitrate, n), bins=25, density=True,
            color='seagreen', alpha=0.7, edgecolor='white')
    ax.set_title(f'Mean of {n} reading(s)')
    ax.set_xlabel('Sample mean (mg/L)')
    ax.set_xlim(0, 9)
plt.tight_layout(); plt.show()

> **Handout Q2.2**

The narrowing follows an exact rule:

$$ SD(\text{sample mean}) = \frac{\sigma}{\sqrt{n}} $$

In [ ]:
pop_sd = np.std(nitrate)

for n in [6, 24, 100]:
    observed  = np.std(sample_means(nitrate, n))
    predicted = pop_sd / np.sqrt(n)
    print(f"n = {n:3d}   simulated SD = {observed:.3f}   σ/√n = {predicted:.3f}")

> **Handout Q2.3 and Q2.4**

---
## Part 3. How Sure Is the Average?

Back to the creek: 30 chloride samples downstream of the treatment plant.

In [ ]:
dn_cl = downstream.column('Cl (mg/L)')

n = downstream.num_rows
sample_mean = np.mean(dn_cl)
sample_sd   = np.std(dn_cl, ddof=1)    # ddof=1: these are a sample, not a population
standard_error = ...

print(f"Number of samples:      {n}")
print(f"Mean chloride:          {sample_mean:.1f} mg/L")
print(f"SD of the measurements: {sample_sd:.1f} mg/L")
print(f"Standard error:         {standard_error:.1f} mg/L")

In [ ]:
# About 95% of a normal distribution lies within 1.96 SDs of the mean
lower = sample_mean - 1.96 * standard_error
upper = ...

print(f"95% confidence interval: {lower:.0f} to {upper:.0f} mg/L")

> **Handout Q3.1, Q3.2, and Q3.3**

---
## Discussion

Your instructor will run the cells below. In Class 15 you built a null distribution by shuffling labels; here is the same test on the Pennypack nitrate data.

In [ ]:
N = pp.column('NO3-N (mg/L)').copy()    # .copy() — shuffling in place would scramble the table!
observed_difference = np.mean(N[9:]) - np.mean(N[:9])

np.random.seed(16)
simulated = []
for i in np.arange(20000):
    shuffled = np.random.permutation(N)
    simulated.append(np.mean(shuffled[9:]) - np.mean(shuffled[:9]))
simulated = np.array(simulated)

plt.figure(figsize=(8, 5))
plt.hist(simulated, bins=40, color='steelblue', alpha=0.7, edgecolor='white')
plt.axvline(observed_difference, color='red', lw=2, label='observed difference')
plt.xlabel('Difference in means under the null hypothesis')
plt.title('20,000 Shuffles'); plt.legend(); plt.show()

hits = np.count_nonzero(simulated >= observed_difference)
print(f"Observed difference: {observed_difference:.3f} mg/L")
print(f"Simulations at least that extreme: {hits} out of 20000")
print(f"p-value = {hits/20000}")

> **Handout D1**

In [ ]:
z = (observed_difference - np.mean(simulated)) / np.std(simulated)
p = 1 - norm.cdf(z)

print(f"Null distribution:  mean = {np.mean(simulated):.3f},  SD = {np.std(simulated):.3f}")
print(f"z-score of the observed difference: {z:.2f}")
print(f"p-value from the normal model: {p:.2e}")

> **Handout D2**